# matmul-back-transpose-pair — ex1: matmul_back — grad_out @ y.T and x.T @ grad_out

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `matmul-back-transpose-pair`. Running the final beacon cell reports progress against the `Backprop: matmul_back transpose pair` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: matmul_back transpose pair` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`matmul-back-transpose-pair`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "matmul-back-transpose-pair"
DD_SUBTOPIC = "Backprop: matmul_back transpose pair"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## `matmul_back` transpose pair — quick refresher

`out = x @ y` (with `x: (m,k)`, `y: (k,n)`, `out: (m,n)`). Both backward fns are themselves matmuls — each contracts `grad_out` with the OTHER input transposed.

**Worked exemplar.** Shape-driven derivation:
```
dL/dx must be (m,k)  →  grad_out @ y.T   :  (m,n) @ (n,k) = (m,k)  ✓
dL/dy must be (k,n)  →  x.T @ grad_out   :  (k,m) @ (m,n) = (k,n)  ✓
```

The transpose-on-the-other-input pattern is general: in `A @ B`, the gradient w.r.t. `A` involves `B.T`, and vice versa.

### Exercise 1 — matmul_back — grad_out @ y.T and x.T @ grad_out

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply the matmul backward pair: derive dL/dx = grad_out @ y.T and dL/dy = x.T @ grad_out from output-shape requirements.
> Keywords: matmul, transpose, shape-derivation, linear
> ```

**KCs targeted:** `matmul-backward-pattern`, `arg-position-back-functions`

Implement two back fns for `out = x @ y` (2-D matmul; `x: (m,k)`, `y: (k,n)`, `out: (m,n)`).

**1. `matmul_back0(grad_out, out, x, y)`** — gradient w.r.t. `x`.
   - `dL/dx` must have shape `(m, k)`.
   - Only one matmul of `grad_out (m,n)` with a transposed input produces that shape: `grad_out @ y.T` is `(m,n) @ (n,k) = (m,k)`. ✓

**2. `matmul_back1(grad_out, out, x, y)`** — gradient w.r.t. `y`.
   - `dL/dy` must have shape `(k, n)`.
   - The shape-matching matmul: `x.T @ grad_out` is `(k,m) @ (m,n) = `(k,n)`. ✓

**The general pattern.** For `A @ B`, the gradient w.r.t. one factor is a matmul that contracts `grad_out` with the OTHER factor transposed. The transpose lives on the input you're NOT differentiating w.r.t.

Use `t.matmul`, `@`, or `.T`. Both inputs are 2-D `torch.Tensor`. Return tensors with the correct shapes. No autograd.

In [ ]:
def matmul_back0(grad_out: Tensor, out: Tensor, x: Tensor, y: Tensor) -> Tensor:
    """dL/dx for out = x @ y. Shape (m, k)."""
    raise NotImplementedError()


def matmul_back1(grad_out: Tensor, out: Tensor, x: Tensor, y: Tensor) -> Tensor:
    """dL/dy for out = x @ y. Shape (k, n)."""
    raise NotImplementedError()


def _test_ex1():
    # --- small (2,3) @ (3,4) = (2,4) ---
    rng = t.Generator().manual_seed(0)
    x = t.randn(2, 3, generator=rng)
    y = t.randn(3, 4, generator=rng)
    out = x @ y
    grad_out = t.randn(2, 4, generator=rng)
    g0 = matmul_back0(grad_out, out, x, y)
    g1 = matmul_back1(grad_out, out, x, y)
    assert g0.shape == (2, 3), f'g0 shape: {g0.shape}'
    assert g1.shape == (3, 4), f'g1 shape: {g1.shape}'
    assert t.allclose(g0, grad_out @ y.T)
    assert t.allclose(g1, x.T @ grad_out)

    # --- square: (3,3) @ (3,3) ---
    x = t.randn(3, 3, generator=rng)
    y = t.randn(3, 3, generator=rng)
    out = x @ y
    grad_out = t.randn(3, 3, generator=rng)
    g0 = matmul_back0(grad_out, out, x, y)
    g1 = matmul_back1(grad_out, out, x, y)
    assert g0.shape == (3, 3) and g1.shape == (3, 3)
    assert t.allclose(g0, grad_out @ y.T)
    assert t.allclose(g1, x.T @ grad_out)

    # --- larger non-square: (5,7) @ (7,3) ---
    x = t.randn(5, 7, generator=rng)
    y = t.randn(7, 3, generator=rng)
    out = x @ y
    grad_out = t.randn(5, 3, generator=rng)
    g0 = matmul_back0(grad_out, out, x, y)
    g1 = matmul_back1(grad_out, out, x, y)
    assert g0.shape == (5, 7)
    assert g1.shape == (7, 3)

    # --- witness vs torch.autograd ---
    x_ref = t.randn(4, 5, requires_grad=True, generator=t.Generator().manual_seed(7))
    y_ref = t.randn(5, 6, requires_grad=True, generator=t.Generator().manual_seed(8))
    z = (x_ref @ y_ref).sum()
    z.backward()
    x_det, y_det = x_ref.detach(), y_ref.detach()
    out_cached = x_det @ y_det
    g0_ours = matmul_back0(t.ones(4, 6), out_cached, x_det, y_det)
    g1_ours = matmul_back1(t.ones(4, 6), out_cached, x_det, y_det)
    assert t.allclose(g0_ours, x_ref.grad, atol=1e-5), (
        f'g0 disagrees with autograd: max diff '
        f'{(g0_ours - x_ref.grad).abs().max()}'
    )
    assert t.allclose(g1_ours, y_ref.grad, atol=1e-5), (
        f'g1 disagrees with autograd: max diff '
        f'{(g1_ours - y_ref.grad).abs().max()}'
    )

    # --- back0 and back1 must be DIFFERENT functions ---
    x_t = t.randn(3, 3, generator=t.Generator().manual_seed(11))
    y_t = t.randn(3, 3, generator=t.Generator().manual_seed(12))
    g_t = t.randn(3, 3, generator=t.Generator().manual_seed(13))
    assert not t.allclose(
        matmul_back0(g_t, x_t @ y_t, x_t, y_t),
        matmul_back1(g_t, x_t @ y_t, x_t, y_t),
    ), 'matmul back0 and back1 must produce different results'
    _dd_passed.add('ex1')
    print("ex1 ✓")

_test_ex1()

<details><summary>Solution</summary>

```python
def matmul_back0(grad_out: Tensor, out: Tensor, x: Tensor, y: Tensor) -> Tensor:
    # Shape (m,n) @ (n,k) = (m,k). y.T transposes y to (n,k).
    return grad_out @ y.T


def matmul_back1(grad_out: Tensor, out: Tensor, x: Tensor, y: Tensor) -> Tensor:
    # Shape (k,m) @ (m,n) = (k,n). x.T transposes x to (k,m).
    return x.T @ grad_out
```

**Shape-driven derivation.** When the math is unfamiliar, derive the backward by shape: grad_out is `(m,n)`, target is `(m,k)` (or `(k,n)`), there's only one matmul that fits. The transpose lives on the input you're NOT differentiating w.r.t.

**Why two separate fns.** Matmul is asymmetric: `back0` involves `y.T`, `back1` involves `x.T`. They're not interchangeable. Registering at both `(matmul, 0)` and `(matmul, 1)` lets the dispatcher route grad_out through the correct one.

**In real autograd,** the same pattern generalizes to batched matmul and einsum — the transpose-on-the-other-input rule is what powers every linear-layer backward in every framework.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()